**Large Language Models**

_Copyright Notice_

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Tool Calling Agent

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

In [2]:
# retrieve API key and other environment variables
load_dotenv()

True

## Normally

In [3]:
from langchain_openai import ChatOpenAI

In [4]:
model = ChatOpenAI(model_name='gpt-6-luna')

In [5]:
from langchain_core.prompts import ChatPromptTemplate

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("human", "{input}"),
    ]
)

In [7]:
msg = prompt.format_prompt(input="전재영의 전화번호를 알려주세요.")

In [8]:
response = model.invoke(msg)

In [9]:
print(response.content)

개인의 전화번호는 알려드릴 수 없습니다. 다만 전재영 씨가 업체나 기관의 공식 연락 담당자라면, 소속이나 관련 정보를 알려주시면 공개된 대표 연락처를 찾는 데 도움을 드릴게요.


## With Tool Calling

### Make Tools

In [10]:
from langchain_core.tools import tool

In [11]:
@tool
def find_phone_number(name: str) -> str:
    """
    주어진 사람의 전화번호를 찾아서 반환합니다.
    전화번호를 찾지 못한 경우 None을 반환합니다.
    """

    phone_book = {
        "전재영": "917-555-0316",
        "홍길동": "718-333-2232",
        "김미선": "613-123-4434",
    }

    if name in phone_book:
        return phone_book[name]
    else:
        return None

### A Collection of Tools

In [12]:
tools = [find_phone_number]

### Set Up Tool Calling Agent

In [13]:
from langchain.agents import create_agent

In [14]:
agent = create_agent(model, tools, system_prompt="You are a helpful assistant")

### Invoke

#### Example 1

In [15]:
response = agent.invoke(
    {
        "messages": [("user", "전재영의 전화번호를 알려주세요.")]
    }
)

In [16]:
response

{'messages': [HumanMessage(content='전재영의 전화번호를 알려주세요.', additional_kwargs={}, response_metadata={}, id='10f1834a-05ea-42bd-989e-006bda62d10b'),
  AIMessage(content=[{'arguments': '{"name":"전재영"}', 'call_id': 'call_m3x1h0wuQwILM6fMFXC47rKV', 'name': 'find_phone_number', 'type': 'function_call', 'id': 'fc_0128fb82bfd9ddaa006af72b6ee43487d2b72bb631b2dca09c', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0128fb82bfd9ddaa006af72b69c45487d2b3444f8787dc8fce', 'created_at': 1794583401.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0128fb82bfd9ddaa006af72b69c45487d2b3444f8787dc8fce', tool_calls=[{'name': 'find_phone_number', 'args': {'name': '전재영'}, 'id': 'call_m3x1h0wuQwILM6fMFXC47rKV', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 21, 'total_tokens': 102, 'input_token_details': 

#### Example 2

In [17]:
response = agent.invoke(
    {
        "messages": [("user", "김미선의 전화번호를 알려주세요.")]
    }
)

In [18]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': '김미선님의 전화번호는 613-123-4434입니다.', 'annotations': [], 'id': 'msg_078f44f19bc44fdb006af72b7e460c87d28c01d60525c5c3b0', 'phase': 'final_answer'}]


#### Example 3

In [19]:
response = agent.invoke(
    {
        "messages": [("user", "박달자의 전화번호를 알려주세요.")]
    }
)

In [20]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': '박달자의 전화번호를 찾지 못했습니다.', 'annotations': [], 'id': 'msg_0ed24e1a97135aa1006af72b83d21c87d2b57f07f628219505', 'phase': 'final_answer'}]
